<a href="https://colab.research.google.com/github/annavorman/laboratory1/blob/main/lab_02_(5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа № 2
## Колоночное хранение, партиционирование и бенчмаркинг Parquet

**Дисциплина:** «Методы обработки больших данных»
**Направление:** 44.03.05 Педагогическое образование
**Профиль:** «Информатика и дополнительное образование (робототехника)»
**Этап конвейера:** `Collect → Store → Process`
**Среда:** Google Colab · PySpark 4.2.0 · `local[*]` · Parquet · Snappy / ZSTD / GZIP

---

### Цель работы

Освоить колоночное хранение телеметрии в формате Parquet, физическое партиционирование набора
и экспериментальное сравнение объёма и времени чтения CSV и Parquet. Понять, за счёт чего
возникает выигрыш и когда он исчезает.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями (`.ipynb`, все ячейки выполнены сверху вниз).
2. Ссылку на ноутбук в GitHub или Colab.
3. Заполненную форму **TEACH CARD** в конце ноутбука.

### Критерии оценки — ровно 10 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | Три физические версии набора: CSV, Parquet, partitioned Parquet | 2 |
| 2 | Размеры, коэффициент сжатия, влияние энтропии и выбора кодека | 2 |
| 3 | Воспроизводимый бенчмарк чтения: ≥ 3 повтора, медиана, честное сравнение | 2 |
| 4 | Column pruning, partition pruning, проблема мелких файлов | 2 |
| 5 | Заполненная форма TEACH CARD | 2 |

> **Правила выполнения.** Не ищите готовые решения — пользуйтесь `help()` и документацией Spark.
> Ячейки с `assert` — самопроверка: если проверка прошла, задание засчитано. Числовые значения
> времени зависят от runtime Colab, поэтому оценивается методика измерения, а не абсолютные цифры.

### Источники

* Balapriya C. `data-science-tutorials` — <https://github.com/balapriyac/data-science-tutorials>
  (`pyspark/pyspark_write_parquet.ipynb`)
* Piotr Szul. `spark-tutorial` — <https://github.com/piotrszul/spark-tutorial>
* Apache Parquet — <https://parquet.apache.org/docs/>
* Spark. Parquet Files — <https://spark.apache.org/docs/latest/sql-data-sources-parquet.html>

---
## Задание 0. Подготовка среды и данных

### Минимальные теоретические сведения

**Колоночное хранение.** Parquet размещает рядом значения одной колонки, а не одной записи.
Отсюда два следствия: однородные данные сжимаются в разы лучше, а запрос, которому нужны две
колонки из десяти, читает только эти две — это **column pruning**.

**Партиционирование.** `partitionBy("robot_id")` раскладывает данные по каталогам вида
`robot_id=robot-7/`. Запрос с фильтром по этой колонке пропускает лишние каталоги целиком —
это **partition pruning**. Значение ключа при этом физически не хранится в файлах: оно
закодировано в имени каталога.

**Метрики работы:**

коэффициент уменьшения объёма  $C = S_{CSV} / S_{Parquet}$

ускорение чтения  $Speedup = T_{CSV} / T_{Parquet}$

Результаты зависят от runtime Colab и файлового кэша, поэтому каждое измерение повторяется
не менее трёх раз, а итогом берётся **медиана** (она устойчивее среднего к единичному выбросу
при первом «холодном» чтении).

In [29]:
!pip -q install "pyspark==4.2.0" pandas pyarrow

In [30]:
import shutil
import time
import statistics
from pathlib import Path

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab02_ParquetBenchmark")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:", spark.version)

ROOT = Path("/content/lab02_storage")
if not Path("/content").exists():          # запуск вне Colab
    ROOT = Path("./lab02_storage")

CSV_DIR         = ROOT / "csv"
PARQUET_DIR     = ROOT / "parquet"
PARTITIONED_DIR = ROOT / "parquet_partitioned"

shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True, exist_ok=True)
print("Каталог хранения:", ROOT)

Spark version: 4.2.0
Каталог хранения: /content/lab02_storage


### Генерация телеметрии

Набор порождается детерминированно, внешний датасет не нужен. Обратите внимание на колонку
`payload`: это SHA-256 от идентификатора, то есть **высокоэнтропийная** строка без повторов.
Она добавлена намеренно — в части 2 мы посмотрим, что она делает со сжатием.

DataFrame кэшируется: иначе Spark пересчитает его заново при каждой из трёх записей.

In [31]:
N = 300_000

df = (
    spark.range(N)
    .withColumn("robot_id",    F.concat(F.lit("robot-"), (F.col("id") % 12).cast("string")))
    .withColumn("sensor",      F.when(F.col("id") % 2 == 0, F.lit("imu")).otherwise(F.lit("motor")))
    .withColumn("ts_ms",       (F.lit(1_720_000_000_000) + F.col("id") * 100).cast("long"))
    .withColumn("temperature", F.lit(40.0) + (F.col("id") % 150) * 0.08)
    .withColumn("vibration",   F.abs(F.sin(F.col("id") / 15.0)) + (F.col("id") % 11) * 0.002)
    .withColumn("payload",     F.sha2(F.col("id").cast("string"), 256))
    .drop("id")
    .cache()
)

print("Строк:", df.count())
df.printSchema()
df.show(3, truncate=False)

Строк: 300000
root
 |-- robot_id: string (nullable = true)
 |-- sensor: string (nullable = false)
 |-- ts_ms: long (nullable = false)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)

+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot_id|sensor|ts_ms        |temperature|vibration          |payload                                                         |
+--------+------+-------------+-----------+-------------------+----------------------------------------------------------------+
|robot-0 |imu   |1720000000000|40.0       |0.0                |5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|
|robot-1 |motor |1720000000100|40.08      |0.068617294923393  |6b86b273ff34fce19d6b804eff5a3f5747ada4eaa22f1d49c01e52ddb7875b4b|
|robot-2 |imu   |1720000000200|40.16      |0.13693862262231407|d4735e3a265e16eee03f59718b9b5d0301

---
# Часть 1. Три физические версии набора (2 балла)

Один и тот же DataFrame нужно записать тремя способами и дальше сравнивать их между собой.

## Упражнение 1.1. CSV

Запишите `df` в CSV в каталог `CSV_DIR`. Режим записи — `overwrite`, заголовок включён
(`option("header", True)`).

In [32]:
# TODO: df.write ... .csv(str(CSV_DIR))
df.write.mode("overwrite").option("header",True).csv(str(CSV_DIR))

files = sorted(p.name for p in CSV_DIR.iterdir())
print("Файлов в каталоге CSV:", len(files))
print(files[:5])
assert CSV_DIR.exists() and any(p.suffix == ".csv" for p in CSV_DIR.iterdir())

Файлов в каталоге CSV: 6
['._SUCCESS.crc', '.part-00000-e989decf-0aee-429d-bf51-ae2bf55fe5d3-c000.csv.crc', '.part-00001-e989decf-0aee-429d-bf51-ae2bf55fe5d3-c000.csv.crc', '_SUCCESS', 'part-00000-e989decf-0aee-429d-bf51-ae2bf55fe5d3-c000.csv']


> **Обратите внимание.** Spark пишет не один файл, а каталог: по одному файлу `part-*` на каждую
> partition плюс маркер `_SUCCESS`. Так устроены все распределённые форматы — единый файл нельзя
> писать параллельно с нескольких узлов.

## Упражнение 1.2. Parquet со сжатием Snappy

Запишите тот же `df` в Parquet в каталог `PARQUET_DIR`, явно указав кодек
`option("compression", "snappy")`.

In [33]:
# TODO: df.write ... .parquet(str(PARQUET_DIR))
df.write.mode("overwrite").option("compression","snappy").parquet(str(PARQUET_DIR))

parts = [p.name for p in PARQUET_DIR.iterdir() if p.suffix == ".parquet"]
print("Файлов parquet:", len(parts))
print(parts[:3])
assert len(parts) > 0
assert all("snappy" in p for p in parts), "В имени файла должен быть виден кодек"

Файлов parquet: 2
['part-00000-db5d60b3-39d2-42ea-b620-aa0805da2c20-c000.snappy.parquet', 'part-00001-db5d60b3-39d2-42ea-b620-aa0805da2c20-c000.snappy.parquet']


## Упражнение 1.3. Partitioned Parquet

Запишите набор в `PARTITIONED_DIR` с физическим партиционированием по `robot_id`
(`partitionBy`). Затем выведите список созданных каталогов и проверьте, что их ровно 12 —
по числу роботов.

In [34]:
# TODO: df.write ... .partitionBy(...).parquet(str(PARTITIONED_DIR))
df.write.mode("overwrite").partitionBy("robot_id").parquet(str(PARTITIONED_DIR))

part_dirs = sorted(p.name for p in PARTITIONED_DIR.iterdir() if p.is_dir())
print("Каталогов:", len(part_dirs))
print(part_dirs[:4])

assert len(part_dirs) == 12
assert all(d.startswith("robot_id=") for d in part_dirs)

Каталогов: 12
['robot_id=robot-0', 'robot_id=robot-1', 'robot_id=robot-10', 'robot_id=robot-11']


## Упражнение 1.4

Прочитайте один файл из каталога `robot_id=robot-7` напрямую (`spark.read.parquet` по полному
пути к каталогу партиции) и посмотрите его схему. Есть ли в ней колонка `robot_id`?
Ответьте письменно, почему.

In [35]:
one_partition = spark.read.parquet(str(PARTITIONED_DIR / "robot_id=robot-7"))
one_partition.printSchema()

ANSWER_1_4 = """
Колонки robot_id в схеме нет. При партиционировании Spark не хранит значение ключа внутри файлов:
оно закодировано в имени каталога robot_id=robot-7. Это само по себе экономит место - строка
"robot-7" не повторяется 25 000 раз, а записана один раз в имени каталога.
Когда мы читаем весь каталог PARTITIONED_DIR целиком, Spark восстанавливает колонку robot_id из имён подкаталогов, и она снова появляется в схеме. Но если читать отдельную партицию
напрямую, восстаналивать не из чего - контекст потерян.
"""
print(ANSWER_1_4)

assert "robot_id" not in one_partition.columns

root
 |-- sensor: string (nullable = true)
 |-- ts_ms: long (nullable = true)
 |-- temperature: double (nullable = true)
 |-- vibration: double (nullable = true)
 |-- payload: string (nullable = true)


Колонки robot_id в схеме нет. При партиционировании Spark не хранит значение ключа внутри файлов:
оно закодировано в имени каталога robot_id=robot-7. Это само по себе экономит место - строка
"robot-7" не повторяется 25 000 раз, а записана один раз в имени каталога.
Когда мы читаем весь каталог PARTITIONED_DIR целиком, Spark восстанавливает колонку robot_id из имён подкаталогов, и она снова появляется в схеме. Но если читать отдельную партицию
напрямую, восстаналивать не из чего - контекст потерян.



---
# Часть 2. Объём хранения (2 балла)

## Упражнение 2.1. Размеры и коэффициент сжатия

Напишите функцию `dir_size(path)`, возвращающую суммарный размер всех файлов каталога в байтах
(каталог вложенный, поэтому нужен рекурсивный обход — подсказка: `Path.rglob`).

Выведите таблицу размеров трёх версий и рассчитайте коэффициент
$C = S_{CSV} / S_{Parquet}$.

In [36]:
def dir_size(path: Path) -> int:
    """Суммарный размер всех файлов внутри каталога, включая вложенные."""
    # TODO
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file())



sizes = {
    "CSV":                 dir_size(CSV_DIR),
    "Parquet":             dir_size(PARQUET_DIR),
    "Partitioned Parquet": dir_size(PARTITIONED_DIR),
}
for name, size in sizes.items():
    print(f"{name:22s}: {size / 1024 / 1024:7.2f} MiB")

compression_ratio = sizes["CSV"] / sizes["Parquet"]
print(f"\nКоэффициент C = S_CSV / S_Parquet = {compression_ratio:.2f}x")

assert sizes["Parquet"] < sizes["CSV"], "Parquet должен быть компактнее CSV"
assert compression_ratio > 1.0

CSV                   :   33.96 MiB
Parquet               :   22.67 MiB
Partitioned Parquet   :   22.85 MiB

Коэффициент C = S_CSV / S_Parquet = 1.50x


## Упражнение 2.2. Энтропия и сжимаемость

Коэффициент из предыдущего упражнения получился скромным. Причина — колонка `payload`:
это SHA-256, то есть практически случайная строка. Данные с высокой энтропией не сжимаются
принципиально: в них нет повторов, которые можно было бы закодировать короче.

Запишите **тот же набор без колонки `payload`** в CSV и в Parquet, посчитайте коэффициент
заново и сравните с предыдущим.

In [37]:
NOPAY_CSV = ROOT / "csv_nopayload"
NOPAY_PQ  = ROOT / "parquet_nopayload"

df_nopay = df.drop("payload")

# TODO: запишите df_nopay в CSV (с заголовком) и в Parquet (snappy)
df_nopay.write.mode("overwrite").option("header", True).csv(str(NOPAY_CSV))
df_nopay.write.mode("overwrite").option("compression","snappy").parquet(str(NOPAY_PQ))

sizes_nopay = {"CSV": dir_size(NOPAY_CSV), "Parquet": dir_size(NOPAY_PQ)}
ratio_nopay = sizes_nopay["CSV"] / sizes_nopay["Parquet"]

print(f"Без payload: CSV {sizes_nopay['CSV']/1024/1024:.2f} MiB, "
      f"Parquet {sizes_nopay['Parquet']/1024/1024:.2f} MiB")
print(f"Коэффициент без payload: {ratio_nopay:.2f}x  (с payload было {compression_ratio:.2f}x)")

ANSWER_2_2 = """
Колонка payload - последовательность, статистически неотличимая от случайной. Её энтропия
близка к максимальной, повторов нет, словарное кодирование бесполезно и Snappy тоже
почти ничего не находит. При этом payload занимает больше половины объема. Эти байты
одинаково плохо сжимаются в обоих форматах и "размывают" выигрыш Parquet.
Остальные колонки устроены противоположным образом: robot_id принимает 12 значений,
sensor-2, temperature - 150. Низкая энтропия, много повторов, идеальный материал для
dictionary encoding и RLE. Как только высокоэнтропийная колонка убрана,
преимущество колоночного формата проявляется в полную силу.
Заявления вида "Parquet сжимает в 10 раз" бессмысленны без указания
структуры данных. Выигрыш опрделеяется энтропией колонок, а не форматом сам по себе.
"""
print(ANSWER_2_2)

assert ratio_nopay > compression_ratio, "Без высокоэнтропийной колонки выигрыш должен вырасти"

Без payload: CSV 15.22 MiB, Parquet 3.61 MiB
Коэффициент без payload: 4.21x  (с payload было 1.50x)

Колонка payload - последовательность, статистически неотличимая от случайной. Её энтропия
близка к максимальной, повторов нет, словарное кодирование бесполезно и Snappy тоже
почти ничего не находит. При этом payload занимает больше половины объема. Эти байты
одинаково плохо сжимаются в обоих форматах и "размывают" выигрыш Parquet.
Остальные колонки устроены противоположным образом: robot_id принимает 12 значений,
sensor-2, temperature - 150. Низкая энтропия, много повторов, идеальный материал для
dictionary encoding и RLE. Как только высокоэнтропийная колонка убрана,
преимущество колоночного формата проявляется в полную силу.
Заявления вида "Parquet сжимает в 10 раз" бессмысленны без указания
структуры данных. Выигрыш опрделеяется энтропией колонок, а не форматом сам по себе.



## Упражнение 2.3. Сравнение кодеков

Запишите `df_nopay` в Parquet четырьмя кодеками: `uncompressed`, `snappy`, `gzip`, `zstd`.
Для каждого измерьте время записи и итоговый размер. Заполните словарь `codec_stats`
в формате `{кодек: (секунды, байты)}`.

In [38]:
codec_stats = {}

for codec in ["uncompressed", "snappy", "gzip", "zstd"]:
    out = ROOT / f"pq_{codec}"
    # TODO: замерьте время записи df_nopay с этим кодеком и размер каталога
    t0=time.perf_counter()
    df_nopay.write.mode("overwrite").option("compression",codec).parquet(str(out))
    elapsed=time.perf_counter()-t0
    codec_stats[codec]=(elapsed,dir_size(out))
    pass

print(f"{'кодек':<14}{'время, с':>10}{'размер, MiB':>14}{'C к uncompressed':>20}")
base = codec_stats["uncompressed"][1]
for codec, (t, s) in codec_stats.items():
    print(f"{codec:<14}{t:>10.2f}{s/1024/1024:>14.2f}{base/s:>20.2f}")

assert set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"}
assert codec_stats["gzip"][1] < codec_stats["snappy"][1], "gzip должен сжимать сильнее snappy"

кодек           время, с   размер, MiB    C к uncompressed
uncompressed        0.62          5.09                1.00
snappy              0.63          3.61                1.41
gzip                0.82          2.70                1.89
zstd                0.63          2.61                1.95


**Вопрос 2.4.** Почему Snappy используется по умолчанию, хотя gzip сжимает заметно сильнее?
Ответьте письменно.

In [39]:
ANSWER_2_4 = """
Выбор кодека - это обмен процессорного времени на объем диска и сети, и Snappy
стоит в точке, удобной для аналитических нагрузок.
Snappy жмёт умеренно, но распаковывается очень быстро - сотни
мегабайт в секунду на ядро.
Файлы Parquet читаются гораздо чаще, чем пишутся, поэтому цена распаковки платится
многократно, и экономить надо именно на ней.
gzip дает заметно меньший объем, но и сжатие, и распаковка стоят
существенно дороже: при аналитическом сканировании
процессор становится узким местом раньше диска.
gzip оправдан для холодного архива, который пишут один раз и читают редко,
Snappy или ZSID  - для данных, по которым регулярно ходят запросы.
"""
print(ANSWER_2_4)


Выбор кодека - это обмен процессорного времени на объем диска и сети, и Snappy
стоит в точке, удобной для аналитических нагрузок.
Snappy жмёт умеренно, но распаковывается очень быстро - сотни
мегабайт в секунду на ядро.
Файлы Parquet читаются гораздо чаще, чем пишутся, поэтому цена распаковки платится
многократно, и экономить надо именно на ней.
gzip дает заметно меньший объем, но и сжатие, и распаковка стоят
существенно дороже: при аналитическом сканировании
процессор становится узким местом раньше диска.
gzip оправдан для холодного архива, который пишут один раз и читают редко,
Snappy или ZSID  - для данных, по которым регулярно ходят запросы.



---
# Часть 3. Бенчмарк чтения (2 балла)

## Упражнение 3.1. Функция измерения

Напишите функцию `benchmark_read(fmt, path, predicate=None, repeats=3, schema=None)`, которая:

1. `repeats` раз читает набор из `path` в формате `fmt` (`"csv"` или `"parquet"`);
2. при заданном `predicate` применяет фильтр;
3. выполняет действие `count()` — **без него из-за ленивости ничего не прочитается**;
4. возвращает кортеж `(медиана времени, число строк)`.

Для CSV используйте `option("header", True)`; параметр `schema` пока оставьте
необязательным — он понадобится в упражнении 3.3.

In [40]:
def benchmark_read(fmt, path, predicate=None, repeats=3, schema=None):
    """Возвращает (медианное время чтения, число строк)."""
    times, counts = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        # TODO: прочитайте данные нужного формата, примените predicate, вызовите count()
        if fmt == "csv":
          reader=spark.read.option("header", True)
          if schema is not None:
            reader = reader.schema(schema)
          else:
            reader=reader.option("inferSchema", True)
          x=reader.csv(str(path))
        else:
            x=spark.read.parquet(str(path))
        if predicate is not None:
         x=x.filter(predicate)
        counts.append(x.count())
        times.append(time.perf_counter()-t0)
    return statistics.median(times), counts[-1]


t, n = benchmark_read("parquet", PARQUET_DIR, repeats=3)
print(f"Чтение всего Parquet: {t:.3f} с, строк {n}")
assert n == N

Чтение всего Parquet: 0.312 с, строк 300000


## Упражнение 3.2. Сравнение трёх версий

Выполните одинаковый запрос `robot_id == "robot-7"` к каждой из трёх версий набора, минимум
по три повтора. Рассчитайте ускорение относительно CSV.

In [41]:
predicate = F.col("robot_id") == "robot-7"

csv_t,  csv_n  = benchmark_read("csv",     CSV_DIR,         predicate)
pq_t,   pq_n   = benchmark_read("parquet", PARQUET_DIR,     predicate)
part_t, part_n = benchmark_read("parquet", PARTITIONED_DIR, predicate)

print(f"CSV                 : {csv_t:6.3f} с, строк {csv_n}")
print(f"Parquet             : {pq_t:6.3f} с, строк {pq_n}")
print(f"Partitioned Parquet : {part_t:6.3f} с, строк {part_n}")
print()
print(f"Speedup CSV -> Parquet             : {csv_t / pq_t:.2f}x")
print(f"Speedup CSV -> Partitioned Parquet : {csv_t / part_t:.2f}x")

assert csv_n == pq_n == part_n == N // 12, "Все три версии должны вернуть одинаковое число строк"

CSV                 :  4.292 с, строк 25000
Parquet             :  0.507 с, строк 25000
Partitioned Parquet :  0.604 с, строк 25000

Speedup CSV -> Parquet             : 8.47x
Speedup CSV -> Partitioned Parquet : 7.11x


## Упражнение 3.3. Честное сравнение

В упражнении 3.1 CSV читался с `inferSchema=True`. Это удобно, но нечестно по отношению к CSV:
чтобы вывести типы, Spark делает **дополнительный полный проход** по файлу. Получается, что
мы сравниваем Parquet с CSV, прочитанным дважды.

Повторите замер CSV с **явно заданной схемой** — возьмите её у уже прочитанного Parquet
(`spark.read.parquet(...).schema`) — и сравните три числа: CSV с выводом типов, CSV со схемой
и Parquet.

In [42]:
explicit_schema = spark.read.parquet(str(PARQUET_DIR)).schema

# TODO: замерьте чтение CSV с явной схемой (schema=explicit_schema)
csv_schema_t, csv_schema_n = benchmark_read("csv",CSV_DIR,predicate,schema=explicit_schema)

print(f"CSV + inferSchema : {csv_t:6.3f} с")
print(f"CSV + явная схема : {csv_schema_t:6.3f} с")
print(f"Parquet           : {pq_t:6.3f} с")
print(f"\nЧестный speedup CSV -> Parquet: {csv_schema_t / pq_t:.2f}x")

ANSWER_3_3 = """
Выигрыш Parquet заметно уменьшился: значительная часть
исходного преимущества объяснялась не форматом,
а лишним проходом по файлу ради вывода типов. С явной схемой
CSV читается за один проход, и сравнение
становится корректным.
Бенчмарк, в котором один вариант поставлен в заведомо худшие условия,
измеряет не то, что заявлено. Если бы мы остановились на первом
замере, то приписали бы колоночному формату чужую заслугу и получили
завышенную оценку выигрыша.
"""
print(ANSWER_3_3)

assert csv_schema_n == csv_n
assert csv_schema_t < csv_t, "Без inferSchema чтение CSV должно ускориться"

CSV + inferSchema :  4.292 с
CSV + явная схема :  0.710 с
Parquet           :  0.507 с

Честный speedup CSV -> Parquet: 1.40x

Выигрыш Parquet заметно уменьшился: значительная часть
исходного преимущества объяснялась не форматом,
а лишним проходом по файлу ради вывода типов. С явной схемой
CSV читается за один проход, и сравнение
становится корректным.
Бенчмарк, в котором один вариант поставлен в заведомо худшие условия,
измеряет не то, что заявлено. Если бы мы остановились на первом
замере, то приписали бы колоночному формату чужую заслугу и получили
завышенную оценку выигрыша.



---
# Часть 4. Pruning и мелкие файлы (2 балла)

## Упражнение 4.1. Partition pruning

Выполните `explain(mode="formatted")` для запроса к partitioned Parquet с фильтром по
`robot_id` и найдите в плане строку `PartitionFilters`.

In [43]:
(
    spark.read.parquet(str(PARTITIONED_DIR))
    .filter(F.col("robot_id") == "robot-7")
    .select("robot_id", "temperature")
    .explain(mode="formatted")
)

== Physical Plan ==
* Project (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [temperature#4047, robot_id#4050]
Batched: true
Location: InMemoryFileIndex [file:/content/lab02_storage/parquet_partitioned]
PartitionFilters: [isnotnull(robot_id#4050), (robot_id#4050 = robot-7)]
ReadSchema: struct<temperature:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [temperature#4047, robot_id#4050]

(3) Project [codegen id : 1]
Output [2]: [robot_id#4050, temperature#4047]
Input [2]: [temperature#4047, robot_id#4050]




In [44]:
ANSWER_4_1 = """
TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?

PartitionFilters содержит условие robot_id=robot-7. Это фильтр по колонке партиционирования:
Spark применяет его к именам каталогов еще до открытия файлов
и читает ровно один каталог из двенадцати. Одиннадцать двенадцатых данных -
не читаются вообще - это partition pruning.
PushedFilters - это условия по обычным колонкам, проталкиваемые
внутрь читателя Parquet.
Они не отменяют чтение файла, но позволяют пропускать row groups
по статистикам min/max в метаданных.
В нашем плане список PushedFilters пуст, потому что единственное условие уже
израсходовано как фильтр партиций.
ReadSchema показывает, какие колонки будут физически
подняты с диска.
В нашем запросе выбрана только temperature, поэтому остальные колонки не читаются. Это column pruning.
"""
print(ANSWER_4_1)


TODO: что содержит PartitionFilters, чем он отличается от PushedFilters,
      и что означает ReadSchema в этом плане?
      
PartitionFilters содержит условие robot_id=robot-7. Это фильтр по колонке партиционирования:
Spark применяет его к именам каталогов еще до открытия файлов
и читает ровно один каталог из двенадцати. Одиннадцать двенадцатых данных -
не читаются вообще - это partition pruning.
PushedFilters - это условия по обычным колонкам, проталкиваемые
внутрь читателя Parquet.
Они не отменяют чтение файла, но позволяют пропускать row groups
по статистикам min/max в метаданных.
В нашем плане список PushedFilters пуст, потому что единственное условие уже
израсходовано как фильтр партиций.
ReadSchema показывает, какие колонки будут физически
подняты с диска.
В нашем запросе выбрана только temperature, поэтому остальные колонки не читаются. Это column pruning.



## Упражнение 4.2. Column pruning в цифрах

Сравните время двух запросов к обычному (непартиционированному) Parquet:

* агрегация **только** по `temperature`;
* агрегация по `temperature` **и** по тяжёлой высокоэнтропийной колонке `payload`.

Второй запрос вынуждает Spark поднять с диска колонку, которая занимает большую часть объёма
файла. Каждый запрос — минимум три повтора, берите медиану.

Дополнительно выведите `explain` обоих вариантов и сравните строки `ReadSchema`.

In [45]:
def timed(fn, repeats=3):
    """Медианное время выполнения fn()."""
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return statistics.median(ts)


def read_with_payload():
    # TODO: agg по temperature И по payload (например, F.max("payload"))
    return (spark.read.parquet(str(PARQUET_DIR))
            .agg(F.avg("temperature"),F.max("payload")).collect())



def read_one_column():
    # TODO: agg только по temperature
    return (spark.read.parquet(str(PARQUET_DIR))
            .agg(F.avg("temperature")).collect())



t_all = timed(read_with_payload)
t_one = timed(read_one_column)
print(f"temperature + payload: {t_all:.3f} с")
print(f"только temperature   : {t_one:.3f} с")
print(f"Column pruning дал ускорение в {t_all / t_one:.2f}x")

spark.read.parquet(str(PARQUET_DIR)).select("temperature").agg(F.avg("temperature")).explain()

ANSWER_4_2 = """
TODO: ваш ответ
Разница во времени объясняется объемом физически прочитанных данных.
В первом запросе нужен максимум по payload - тяжелой
высокоэнтропийной колонке, которая занимает
большую часть объема файла, поэтому она поднимается с диска целиком.
Во втором запросе payload не упоминается, и Parquet ее вообще не читает:
значения каждой колонки лежат отдельным непрерывным куском,
и ненужные куски просто пропускаются.
Строка ReadSchema сужается до одной колонки даже без явного select.
Catalyst сам определяет по запросу, какие колонки нужны, и проталкивает
этот список в читатель Parquet.
То есть column pruning выполняется автоматически, и попытка "помочь" оптимизатору
лишним select ничего не изменит - в отличие от партиционирования, которое надо
спроектировать заранее руками.
Избегать SELECT * важно не потому, что Spark не догадается, а потому, что каждая реально
запрошенная тяжёлая колонка стоит времени. На широких таблицах разница между
"нужны две колонки из пятидесяти" и "нужны все пятьдесят" - это разница
в объеме дискового ввода-вывода на порядок.
"""
print(ANSWER_4_2)

temperature + payload: 0.882 с
только temperature   : 0.274 с
Column pruning дал ускорение в 3.22x
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(temperature#4211)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=2835]
      +- HashAggregate(keys=[], functions=[partial_avg(temperature#4211)])
         +- FileScan parquet [temperature#4211] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/lab02_storage/parquet], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<temperature:double>



TODO: ваш ответ
Разница во времени объясняется объемом физически прочитанных данных.
В первом запросе нужен максимум по payload - тяжелой 
высокоэнтропийной колонке, которая занимает
большую часть объема файла, поэтому она поднимается с диска целиком. 
Во втором запросе payload не упоминается, и Parquet ее вообще не читает: 
значения каждой колонки лежат отдельным непрерывным куском,

## Упражнение 4.3. Проблема мелких файлов

Партиционирование выглядит бесплатным улучшением, но это не так. Запишите набор,
партиционированный по искусственному ключу высокой кардинальности `ts_bucket` (200 значений),
и сравните с партиционированием по `robot_id` (12 значений):

* число созданных файлов;
* суммарный объём;
* средний размер одного файла;
* время чтения всего набора.

In [46]:
MANY_DIR = ROOT / "parquet_many_partitions"

df_buckets = df_nopay.withColumn(
    "ts_bucket", ((F.col("ts_ms") / 100) % 200).cast("int")
)
# TODO: запишите df_buckets в MANY_DIR с partitionBy("ts_bucket")
df_buckets.write.mode("overwrite").partitionBy("ts_bucket").parquet(str(MANY_DIR))


def file_stats(path: Path):
    """Возвращает (число файлов .parquet, суммарный размер, средний размер)."""
    # TODO
    files=[p for p in path.rglob("*.parquet") if p.is_file()]
    total=sum(p.stat().st_size for p in files)
    return len(files), total, total / max(len(files),1)
    raise NotImplementedError


n_few,  s_few,  avg_few  = file_stats(PARTITIONED_DIR)
n_many, s_many, avg_many = file_stats(MANY_DIR)

print(f"{'вариант':<28}{'файлов':>9}{'объём, MiB':>13}{'средний файл, KiB':>20}")
print(f"{'partitionBy(robot_id) [12]':<28}{n_few:>9}{s_few/1024/1024:>13.2f}{avg_few/1024:>20.1f}")
print(f"{'partitionBy(ts_bucket) [200]':<28}{n_many:>9}{s_many/1024/1024:>13.2f}{avg_many/1024:>20.1f}")

t_few  = timed(lambda: spark.read.parquet(str(PARTITIONED_DIR)).count())
t_many = timed(lambda: spark.read.parquet(str(MANY_DIR)).count())
print(f"\nЧтение целиком: 12 партиций {t_few:.3f} с, 200 партиций {t_many:.3f} с")

assert n_many > n_few
assert avg_many < avg_few

вариант                        файлов   объём, MiB   средний файл, KiB
partitionBy(robot_id) [12]         24        22.67               967.5
partitionBy(ts_bucket) [200]      400         4.47                11.4

Чтение целиком: 12 партиций 0.323 с, 200 партиций 3.892 с


## Упражнение 4.4. Технический вывод

Ответьте письменно:

1. Почему слишком большое число мелких partition ухудшает эффективность хранения и чтения?
2. По какому признаку выбирать колонку для `partitionBy`?
3. Когда партиционирование по `robot_id` в нашей задаче окажется бесполезным?

In [47]:
ANSWER_4_4 = """
TODO: ваш ответ (7-10 предложений)
1. Мелкие файлы плохи сразу по нескольким причинам. Parquet - формат с заметными накладными расходами: у каждого файла есть footer со схемой
и статистиками, словари кодирования, метаданные row groups. На файле в 10 Мб это доли процента, на файле в 50 Кб - заметная доля объема, и
суммарный размер набора растёт. Словарь и RLE  работают тем лучше, чем длиннее непрерывный участок однородных данных, а в крошечном файле его просто нет.
При чтении каждый файл - это отдельное открытие, чтение footer и планирование задачи; на тысячах файлов время уходит на оркестрацию, а не на данные. В реальном HDFS
добавляется проблема: метаданные всех файлов держит в памяти NameNode, и миллионы мелких файлов его переполняют.
2. Колонка для partitionBy должна удовлетворять трём условиям: по ней часто фильтруют (иначе pruning не сработает и все издержки будут напрасны),
у нее умеренная кардинальность (десятки или сотни значений, а не миллионы), и данные распределены по её значениям более или менее
удачный выбор - дата; классический неудачный - идентификатор события или временная метка с точностью до миллисекунды.
3. Партиционирование по robot_id бесполезно и даже вредно, если запросы никогда не фильтруют по роботу,
а всегда идут по всему парку - например, "средняя температура по всем роботам за сутки".
Тогда читаются все каталоги, pruning не срабатывает, а накладные расходы на дробление остаются.
Также онон перестанет работать при сильном перекосе: если один робот даёт половину записей, его партиция станет огромной, остальные крошечными,
и параллелизм ухудшится. И наконец, если роботов станет не 12, а 50 000, кардинальность окажется слишком высокой - понадобится
партиционирование по дате с дополнительной кластеризацией внутри партиции.
"""
print(ANSWER_4_4)


TODO: ваш ответ (7-10 предложений)
1. Мелкие файлы плохи сразу по нескольким причинам. Parquet - формат с заметными накладными расходами: у каждого файла есть footer со схемой 
и статистиками, словари кодирования, метаданные row groups. На файле в 10 Мб это доли процента, на файле в 50 Кб - заметная доля объема, и
суммарный размер набора растёт. Словарь и RLE  работают тем лучше, чем длиннее непрерывный участок однородных данных, а в крошечном файле его просто нет.
При чтении каждый файл - это отдельное открытие, чтение footer и планирование задачи; на тысячах файлов время уходит на оркестрацию, а не на данные. В реальном HDFS 
добавляется проблема: метаданные всех файлов держит в памяти NameNode, и миллионы мелких файлов его переполняют. 
2. Колонка для partitionBy должна удовлетворять трём условиям: по ней часто фильтруют (иначе pruning не сработает и все издержки будут напрасны),
у нее умеренная кардинальность (десятки или сотни значений, а не миллионы), и данные распределены по её

---
# Часть 5. TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Это **обязательная**
часть работы.

**Требования к заполнению.** Формулировки конкретные. «Расскажу про Parquet» не засчитывается.
Нужны измеримый результат, критерий успеха и честно названные ограничения.

**Подсказка по содержанию.** Эксперимент «CSV против Parquet» удобен для школы тем, что результат
виден глазами: два каталога с одними и теми же данными занимают разный объём, и это можно
измерить без единой строки распределённого кода — хватит `du -h`. Аналогия для урока: если
нужна только температура, читается только «стопка температур», а не все листы журнала целиком.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | CSV против Parquet: как формат хранения влияет на объём и скорость чтения телеметрии робота |
| **Целевая аудитория** | кружок робототехники |
| **Исследовательский вопрос** | Во сколько раз формат Parquet с партицированием по robot_id уменьшает объём файла и ускоряет чтение по сравнению с CSV при выборке телеметрии одного робота из набора в 300 000 записей? |
| **Источник данных** | Синтетически сгенерированный DataFrame PySpark (300 000 строк, 6 колонок: robot_id, sensor, ts_ms, temperature, vibration, payload-SHA256)|
| **Инженерная концепция, сохраняемая без упрощения** | Колоночное хранение (column pruning): Parquet физически размещает значения одной колонки рядом, поэтому запрос, читающий 2 колонки из 6, не загружает остальные 4. Партицирование раскладывает данные по каталогам вида robot_id=robot-7/, и фильтр по этой колонке пропускает лишние каталоги целиком (partition pruning). |
| **Что упрощается относительно университетской лабораторной** | 1) Нет кластера Spark. 2)Запись идёт в локальную файловую систему Colab. 3) Объём данных на порядки меньше промышленного, но достаточен для демонстрации эффектов. |
| **Алгоритм действий обучающегося (4–6 шагов)** | 1. Запустить ячейку генерации DataFrame (300 000 строк) и убедиться, что схема содержит 6 колонок, включая высокоэнтропийную payload. 2. Сохранить DataFrame в трёх физических версиях: CSV, Parquet (Snappy), Parquet с partitionBy("robot_id"). 3. Измерить размер каждого каталога командой du -h и вычислить коэффициент сжатия C = S_CSV / S_Parquet. 4. Провести бенчмарк чтения: 3 повтора запроса SELECT temperature, vibration FROM … WHERE robot_id='robot-7', взять медиану времени, вычислить Speedup = T_CSV / T_Parquet. 5. Повторить шаги 3–4 для кодеков ZSTD и GZIP, сравнить результаты в таблице. |
| **Измеримый результат / метрика** |1. Размер каталога CSV, Parquet/Snappy, partitioned Parquet/Snappy в байтах. 2. Коэффициент уменьшения объёма C = S_CSV / S_Parquet. 3. Медианное время чтения (мс) по 3 повторам для CSV и Parquet. 4. Ускорение чтения Speedup = T_CSV / T_Parquet. 5. Таблица сравнения кодеков Snappy / ZSTD / GZIP по объёму и времени.  |
| **Критерий успешного выполнения** | C ≥ 2 для Parquet vs CSV на «температурных» колонках; Speedup ≥ 1.5 при чтении одного робота из партицированного Parquet. |
| **Вариант усложнения для НТО / хакатона** |Построить график зависимости C и Speedup от числа строк (10K → 1M) и найти точку, где Parquet перестаёт выигрывать. 1) Добавить колонку с реальными JSON-полезными нагрузками (не SHA256) и сравнить сжатие.  |
| **Риски и ограничения** |1) Числовые значения времени зависят от runtime Colab и файлового кэша — оценивается методика измерения, а не абсолютные цифры. 2) Высокоэнтропийная колонка payload (SHA-256) почти не сжимается — это намеренно, чтобы показать предел выигрыша Parquet. 3) В Colab нет распределённого исполнения — результаты не масштабируются напрямую на кластер.  |

---
## Итоговая самопроверка

In [48]:
checks = {
    "1.1 CSV записан":          CSV_DIR.exists(),
    "1.2 Parquet записан":      len(list(PARQUET_DIR.glob("*.parquet"))) > 0,
    "1.3 partitionBy":          len(part_dirs) == 12,
    "1.4 схема партиции":       "TODO" not in ANSWER_1_4,
    "2.1 размеры и C":          compression_ratio > 1.0,
    "2.2 энтропия":             ratio_nopay > compression_ratio and "TODO" not in ANSWER_2_2,
    "2.3 кодеки":               set(codec_stats) == {"uncompressed", "snappy", "gzip", "zstd"},
    "2.4 выбор кодека":         "TODO" not in ANSWER_2_4,
    "3.2 бенчмарк чтения":      csv_n == pq_n == part_n,
    "3.3 честное сравнение":    csv_schema_t < csv_t and "TODO" not in ANSWER_3_3,
    "4.1 partition pruning":    "TODO" not in ANSWER_4_1,
    "4.2 column pruning":       t_all > t_one and "TODO" not in ANSWER_4_2,
    "4.3 мелкие файлы":         n_many > n_few,
    "4.4 технический вывод":    "TODO" not in ANSWER_4_4,
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
print("TEACH CARD проверяется преподавателем вручную.")

OK   1.1 CSV записан
OK   1.2 Parquet записан
OK   1.3 partitionBy
OK   1.4 схема партиции
OK   2.1 размеры и C
OK   2.2 энтропия
OK   2.3 кодеки
OK   2.4 выбор кодека
OK   3.2 бенчмарк чтения
OK   3.3 честное сравнение
НЕТ  4.1 partition pruning
НЕТ  4.2 column pruning
OK   4.3 мелкие файлы
НЕТ  4.4 технический вывод

Выполнено: 11 из 14
TEACH CARD проверяется преподавателем вручную.


In [49]:
df.unpersist()
spark.stop()
print("SparkSession остановлена.")

SparkSession остановлена.
